# Tydzień 5: Klasyfikacja – eksploracja danych i `setup()` 🌸

| | |
|---|---|
| **Rozdział książki** | 3. *Classification* – wstęp (regresja logistyczna), *Loading the Dataset*, *Exploratory Data Analysis*, *Initializing the PyCaret Environment* |
| **Czas zajęć** | 90 minut |
| **Dane** | `iris` – pomiary 150 kwiatów irysa (wbudowane w PyCaret) |
| **Nowe funkcje** | `sns.boxplot`, `sns.heatmap`, `sns.pairplot`, `setup()` dla klasyfikacji |

## 🎯 Cel zajęć
Po dzisiejszych zajęciach będziesz:
1. rozumieć, czym jest **klasyfikacja** (*classification*) i czym różni się od regresji,
2. znać ideę **regresji logistycznej** (*logistic regression*) – mimo nazwy to model do klasyfikacji!,
3. umieć przeprowadzić EDA dla klasyfikacji: sprawdzić **zbalansowanie klas**, porównać cechy między klasami,
4. umieć przygotować dane funkcją `setup()` i zrozumieć **kodowanie etykiet** (*label encoding*).

## 🗓️ Plan zajęć
| Czas | Co robimy |
|---|---|
| 0–20 min | 📚 Teoria: klasyfikacja i regresja logistyczna |
| 20–65 min | 👣 Krok po kroku: EDA irysów + `setup()` |
| 65–85 min | 🧑‍💻 Ćwiczenie samodzielne – ✅ **obowiązkowe** (zaliczenie) |
| 85–90 min | 📝 Podsumowanie |
| poza zajęciami | 🏋️ Część 4: ćwiczenia dodatkowe (trudniejsze, nieobowiązkowe) – dla szybszych osób lub do domu |

---
# 📚 Część 1: Wprowadzenie teoretyczne

## Co to jest klasyfikacja? 🗂️
W **klasyfikacji** przewidujemy **kategorię** (klasę, *class label*) – tak jakbyśmy wrzucali listy do właściwych
przegródek na poczcie.
- **Klasyfikacja binarna** (*binary*): dwie klasy, np. spam / nie spam, chory / zdrowy (często zapisywane jako 1 / 0).
- **Klasyfikacja wieloklasowa** (*multiclass*): więcej klas, np. trzy gatunki irysa.

| | Regresja | Klasyfikacja |
|---|---|---|
| Co przewidujemy? | liczbę (np. 13 270 $) | kategorię (np. *Iris-setosa*) |
| Przykładowa metryka | RMSE, R² | dokładność (*accuracy*) |

## Regresja logistyczna – „termometr prawdopodobieństwa” 🌡️
Mimo nazwy **regresja logistyczna** służy do **klasyfikacji**. Działa w dwóch krokach:
1. Liczy „punkty” jak regresja liniowa: $z = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$ (wynik może być dowolną liczbą).
2. Zamienia punkty na **prawdopodobieństwo** od 0 do 1 za pomocą funkcji logistycznej (sigmoidy), która ma kształt litery „S”:

$$p = \sigma(z) = \frac{1}{1 + e^{-z}}$$

Bardzo dużo punktów → p bliskie 1 (prawie na pewno klasa 1). Bardzo mało → p bliskie 0. Gdy **p > 0,5**, wybieramy klasę 1.
To jak ściemniacz światła: zamiast „włączone / wyłączone” mamy płynną skalę, a próg 0,5 decyduje o odpowiedzi.

Oprócz regresji logistycznej są inne klasyfikatory: drzewa decyzyjne, k-najbliższych sąsiadów (KNN),
liniowa analiza dyskryminacyjna (LDA), XGBoost… Porównamy je za tydzień.

## Zbiór Iris 🌸
Słynny zbiór danych zebrany w latach 30. XX wieku (opracowany statystycznie przez Ronalda Fishera). 150 kwiatów, po 50 z każdego gatunku:
*Iris setosa*, *Iris versicolor*, *Iris virginica*. Cechy (w centymetrach):
- `sepal_length`, `sepal_width` – długość i szerokość **działki kielicha** (zielony „listek” pod kwiatem),
- `petal_length`, `petal_width` – długość i szerokość **płatka**.

## Zbalansowane klasy ⚖️
Jeśli 95% e-maili to nie-spam, „leniwy” model, który zawsze mówi „nie spam”, ma 95% trafności – a jest bezużyteczny!
Dlatego w EDA sprawdzamy, czy klasy są **zbalansowane** (podobnie liczne). W irysach są idealnie równe – po 33,3%.

---
# 👣 Część 2: Krok po kroku

### Krok 1: Importy i dane

In [ ]:
# Wyłączamy ostrzeżenia (warnings), żeby nie zaśmiecały ekranu.
# Ostrzeżenie to NIE błąd – to tylko podpowiedź dla programistów bibliotek.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Biblioteki do tabel i wykresów
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# get_data – dane; "import *" z modułu KLASYFIKACJI PyCaret (inny moduł niż w regresji!)
from pycaret.datasets import get_data
from pycaret.classification import *

In [ ]:
# Wczytujemy zbiór irysów
data = get_data('iris')

In [ ]:
# Podstawowe informacje: 150 wierszy, 4 kolumny liczbowe (float64) i 1 tekstowa (object) – gatunek
data.info()

In [ ]:
# Statystyki opisowe cech liczbowych (w cm)
data.describe()

### Krok 2: Czy klasy są zbalansowane? 🥧

In [ ]:
# Liczba kwiatów w każdej klasie
data['species'].value_counts()

In [ ]:
# Wykres kołowy (pie chart) proporcji klas; autopct dopisuje procenty
data['species'].value_counts().plot(kind='pie', autopct='%.1f%%')
plt.ylabel('')   # usuwamy zbędny podpis osi
plt.show()

🔎 Każdy gatunek to dokładnie 33,3% danych – klasy są **idealnie zbalansowane**. Dzięki temu możemy spokojnie używać
metryki **dokładność** (*accuracy*). Przy niezbalansowanych danych PyCaret oferuje m.in. parametr `fix_imbalance=True`
(użyjemy go w tygodniu 9).

### Krok 3: Czym różnią się gatunki? – wykresy pudełkowe 📦

In [ ]:
# Wykres pudełkowy długości płatka dla każdego gatunku
sns.boxplot(data=data, x='species', y='petal_length')
plt.show()

In [ ]:
# To samo dla szerokości działki kielicha
sns.boxplot(data=data, x='species', y='sepal_width')
plt.show()

🔎 **Długość płatka** świetnie rozróżnia gatunki – pudełka w ogóle na siebie nie zachodzą (szczególnie *setosa*).
**Szerokość działki** rozróżnia je dużo słabiej – pudełka się nakładają.

💡 Zamiast rysować wykresy jeden po drugim, możemy narysować wszystkie naraz w pętli (tak jak w książce):

In [ ]:
# Siatka 2×2 wykresów; pętla for rysuje po jednym wykresie pudełkowym dla każdej z 4 cech
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, col in zip(axes.flatten(), data.columns[:4]):
    sns.boxplot(data=data, x='species', y=col, ax=ax)
plt.show()

In [ ]:
# Średnia wartość każdej cechy w każdym gatunku – liczby potwierdzają to, co widać na wykresach
data.groupby('species').mean()

### Krok 4: Korelacje między cechami 🌡️

In [ ]:
# numeric_only=True – liczymy korelację tylko dla kolumn liczbowych (pomijamy tekstową kolumnę species)
corr = data.corr(numeric_only=True)

In [ ]:
# Mapa ciepła korelacji; annot=True wpisuje wartości w kratki
sns.heatmap(corr.round(2), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

🔎 Długość i szerokość płatka są **bardzo silnie skorelowane** (0,96): długie płatki są też szerokie.
Szerokość działki kielicha jest słabo, **ujemnie** skorelowana z pozostałymi cechami.

### Krok 5: Macierz wykresów rozrzutu 🔵🟠🟢

In [ ]:
# pairplot: wykres rozrzutu dla każdej pary cech, kolory = gatunki; na przekątnej rozkłady cech
sns.pairplot(data, hue='species')
plt.show()

🔎 **Najważniejsza obserwacja:** *Iris setosa* (niebieska) jest **całkowicie oddzielona** od pozostałych gatunków –
łatwo ją rozpoznać. *Versicolor* i *virginica* trochę na siebie **nachodzą** – tu model może się czasem pomylić.

### Krok 6: Inicjalizacja środowiska PyCaret – `setup()` ⚙️

In [ ]:
classf = setup(
    data=data,           # dane
    target='species',    # zmienna celu – gatunek
    train_size=0.8,      # 80% trening, 20% test
    normalize=True,      # normalizacja (z-score) – pomaga np. modelom KNN i LDA
    session_id=3934,     # ziarno losowości – powtarzalne wyniki
)

📖 **Nowe wiersze w tabeli (w porównaniu z regresją):**
- `Target type` = **Multiclass** – PyCaret sam rozpoznał klasyfikację wieloklasową,
- `Target mapping` – **kodowanie etykiet** (*label encoding*): modele wolą liczby, więc nazwy gatunków
  zamieniono na 0, 1, 2 (`Iris-setosa: 0, Iris-versicolor: 1, Iris-virginica: 2`),
- `Fold Generator` = **StratifiedKFold** – walidacja krzyżowa **warstwowa** (*stratified*): w każdej części zachowane są
  proporcje klas (po ⅓ każdego gatunku), tak jak w dobrze przygotowanej ankiecie reprezentatywnej.

In [ ]:
# Rozmiary zbiorów: 120 kwiatów do treningu, 30 do testu
print("Trening:", get_config('X_train').shape)
print("Test:   ", get_config('X_test').shape)

In [ ]:
# Dane treningowe po normalizacji – wartości wokół 0
get_config('X_train_transformed').head()

In [ ]:
# Zmienna celu po zakodowaniu – liczby 0, 1, 2 zamiast nazw gatunków
get_config('y_train_transformed').value_counts()

🔎 W zbiorze treningowym jest po 40 kwiatów każdego gatunku – podział zachował proporcje klas
(PyCaret domyślnie dzieli dane **warstwowo**, *stratified split*).

---
# 🧑‍💻 Część 3: Ćwiczenie dla Ciebie
> ✅ **Ćwiczenie obowiązkowe** – na jego podstawie zaliczasz dzisiejsze zajęcia (zaliczone / niezaliczone).

**Zadanie:** Kontynuuj eksplorację irysów.

- **a)** Narysuj histogram `petal_width` z podziałem kolorem na gatunki (`hue='species'`). Który gatunek ma najwęższe płatki?
- **b)** Narysuj wykres rozrzutu `petal_length` (oś X) vs `petal_width` (oś Y) z kolorami gatunków.
  Czy da się „narysować kreskę” oddzielającą *setosę* od reszty?
- **c)** Uruchom nowy `setup()` z podziałem **70% / 30%** (`train_size=0.7`) i sprawdź, ile kwiatów trafiło do zbioru testowego.

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- a) `sns.histplot(data=data, x='...', hue='...')`
- b) `sns.scatterplot(data=data, x='...', y='...', hue='...')`
- c) Skopiuj komórkę z `setup()` i zmień jeden parametr; potem `get_config('X_test').shape`.
  Dodaj `verbose=False`, żeby ukryć tabelę.

</details>

In [ ]:
# ✍️ Twój kod tutaj (możesz dodawać kolejne komórki przyciskiem "+ Code")

### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 🏋️ Część 4: Ćwiczenia dodatkowe – trochę trudniejsze
Dla osób, które skończyły ćwiczenie podstawowe, albo **do samodzielnej pracy w domu**.
Ćwiczenia z tej części są **nieobowiązkowe** – nie są wymagane do zaliczenia zajęć.

Te zadania mają mniej podpowiedzi w treści, łączą kilka poznanych narzędzi i czasem wprowadzają **jedną nową funkcję**
(zawsze opisaną w podpowiedzi). Poziomy trudności: ⭐⭐ – średnie, ⭐⭐⭐ – wyzwanie.

## ⭐⭐ Ćwiczenie 2: Nowe cechy – powierzchnia płatka i działki
Czasem **połączenie** dwóch cech rozdziela klasy lepiej niż każda z nich osobno.

- **a)** Dodaj do `data` dwie nowe kolumny: `petal_area` = długość × szerokość płatka oraz `sepal_area` = długość × szerokość działki kielicha
  (przybliżona „powierzchnia” w cm²).
- **b)** Narysuj wykres pudełkowy `petal_area` dla każdego gatunku.
- **c)** Dla każdego gatunku policz **minimum, maksimum i średnią** obu nowych cech.
- **d)** Czy na podstawie samej `petal_area` da się bezbłędnie rozpoznać gatunek? A na podstawie `sepal_area`?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Mnożenie kolumn działa „wiersz po wierszu”: `data['petal_length'] * data['petal_width']`.
- Kilka statystyk naraz: `data.groupby('species')[['petal_area', 'sepal_area']].agg(['min', 'max', 'mean'])`.
- Klasy da się rozdzielić bezbłędnie, jeśli zakresy (min–max) gatunków **nie nachodzą** na siebie.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

⚠️ Przed ćwiczeniem 3 usuwamy kolumny dodane w ćwiczeniu 2, żeby nie zmieniały danych w kolejnych krokach.
Uruchom komórkę poniżej (zadziała także wtedy, gdy tych kolumn nie dodano):

In [ ]:
# Sprzątanie: usuwamy kolumny z ćwiczenia 2 (errors='ignore' – brak błędu, jeśli kolumn nie ma)
data = data.drop(columns=['petal_area', 'sepal_area'], errors='ignore')

## ⭐⭐⭐ Ćwiczenie 3: Po co podział warstwowy?
PyCaret domyślnie dzieli dane **warstwowo** (*stratified*) – zachowuje proporcje klas w zbiorze treningowym i testowym.
Sprawdźmy, co by się stało bez tego.

- **a)** Uruchom `setup()` z `train_size=0.7`, `session_id=1` i policz, ile kwiatów każdego gatunku trafiło do **zbioru testowego**.
- **b)** Powtórz to samo z dodatkowym parametrem `data_split_stratify=False` (podział **całkowicie losowy**).
- **c)** Porównaj wyniki. Dlaczego przy małych zbiorach danych podział warstwowy jest ważny?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Zbiór testowy zmiennej celu: `get_config('y_test')`, a liczebność klas: `.value_counts()`.
- Dodaj `verbose=False`, żeby nie wyświetlać tabel z `setup()`.
- Pomyśl o skrajnym przypadku: co, gdyby do zbioru testowego trafiło tylko 5 kwiatów jednego gatunku, a 20 innego?

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 📝 Podsumowanie
- **Klasyfikacja** przewiduje kategorię: binarną (2 klasy) lub wieloklasową (więcej klas).
- **Regresja logistyczna** liczy prawdopodobieństwo klasy funkcją w kształcie „S” (sigmoidą).
- W EDA klasyfikacji sprawdzamy **zbalansowanie klas** i to, które cechy **rozdzielają klasy** (wykresy pudełkowe, `pairplot`).
- `setup()` dla klasyfikacji automatycznie: koduje etykiety (*label encoding*), dzieli dane warstwowo i używa `StratifiedKFold`.
- W irysach najlepiej rozróżniają gatunki **wymiary płatka**; *setosa* jest całkowicie oddzielona od reszty.

## 📖 Słowniczek PL / EN
| Polski | English | Znaczenie |
|---|---|---|
| klasyfikacja binarna / wieloklasowa | binary / multiclass classification | 2 klasy / więcej klas |
| klasa, etykieta klasy | class, class label | kategoria, którą przewidujemy |
| regresja logistyczna | logistic regression | model klasyfikacji oparty na sigmoidzie |
| funkcja logistyczna (sigmoida) | logistic (sigmoid) function | zamienia dowolną liczbę na wartość 0–1 |
| prawdopodobieństwo | probability | szansa przynależności do klasy |
| zbalansowane klasy | balanced classes | klasy o podobnej liczebności |
| kodowanie etykiet | label encoding | zamiana nazw klas na liczby 0, 1, 2… |
| podział warstwowy | stratified split / StratifiedKFold | zachowanie proporcji klas w każdej części |
| wykres kołowy | pie chart | udział każdej kategorii w całości |

➡️ **Za tydzień:** porównamy klasyfikatory, poznamy **macierz pomyłek** (*confusion matrix*) i narysujemy **granice decyzyjne**.